# 09l Alljoined-1.6M test P-6: the personal map on consumer EEG (prespecified, scored once)

**Kaggle:** CPU, Internet on. **Inputs:** the Alljoined dataset (09k), the THINGS-EEG2 dataset (09g, only for the
margin) and the outputs of 03b.

Protocol P-6 (`docs/proposals/alljoined_confirmation_protocol.md`). 16 people (test reliability at least 0.05), each
borrowing from the other 15; 0 to 790 ms; k = 250 and 500, draws 13 to 15. Claims, Bonferroni over three at alpha
0.05: C1 the map beats borrowing; C2 the map improves Gu's ensemble; C3 the map beats Procrustes per window. Margin
0.011 times the ratio of Alljoined's to THINGS-EEG2's median test reliability, computed before any prediction is
scored. All 20 people are repeated as a descriptive secondary.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09l-alljoined', 'cpu-confirm'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"included": [1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 17, 18, 20], "all": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20], "budgets": [250, 500], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "crop_s": [0.0, 0.79], "n_samples": 80, "r_test_primary": 79, "alpha": 0.05, "min_people": 14, "eeg2_delta": 0.011, "ceiling_splits": 20, "seed": 13, "oof_folds": 5, "map_window_ms": 100, "eeg2_people": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"], "claims": {"C1": ["h4b", "avg"], "C2": ["gu+map", "gu"], "C3": ["h4b", "procrustes"]}, "protocol": "P-6 frozen: Alljoined-1.6M confirmation, new family, alpha 0.05 Bonferroni over 3 claims"}')
import numpy as np
import pandas as pd
from scipy import stats

from cross_subject_encoding.data import alljoined as aj
from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import alignment_controls as al
from cross_subject_encoding.eval import eeg1_decision as dec
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval import plugin_map as pm
from cross_subject_encoding.eval.noise_ceiling import noise_ceiling
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.baselines import gu_by_group, window_groups
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

def find(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    if len(hits) != 1:
        raise FileNotFoundError(f"{name}: {len(hits)} matches under /kaggle/input")
    return hits[0]

with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
N_TRAIN, N_TEST = len(z_train), len(z_test)
y_train, bio, n_rep_min, times = {}, {}, {}, None
for p in FIXED["all"]:
    sub = f"sub-{p:02d}"
    tr = te.load_part(find(aj.FLAT.format(subject=sub, part="training")))
    ts = te.load_part(find(aj.FLAT.format(subject=sub, part="test")))
    # half a sample (5 ms) of tolerance: the release's grid starts 0.8 ms after onset, so samples sit at 0.001, 0.011, ... s
    crop = (tr["times"] >= FIXED["crop_s"][0] - 0.005) & (tr["times"] < FIXED["crop_s"][1] + 0.005)
    if crop.sum() != FIXED["n_samples"] or not np.allclose(ts["times"][crop], tr["times"][crop]):
        raise ValueError(f"{sub}: crop gives {crop.sum()} samples")
    t_p = np.round(tr["times"][crop], 3)
    times = t_p if times is None else times
    if not np.allclose(t_p, times):
        raise ValueError(f"{sub}: cropped times differ from the first person's")
    y_train[p] = np.nanmean(tr["data"][..., crop], axis=1).astype(np.float32)  # mean over available repetitions
    test = ts["data"][..., crop]
    n_rep_min[p] = int((~np.isnan(test[:, :, 0, 0])).sum(1).min())
    bio[p] = test
    del tr, ts
    if not np.isfinite(y_train[p]).all():
        raise ValueError(f"{sub}: an image has no training repetition")
print("loaded 20 people, times", times[0], "to", times[-1], "| min test reps", n_rep_min, "|", f"{(time.monotonic() - T0) / 60:.1f} min")
R = min(n_rep_min[p] for p in FIXED["included"])
assert R == FIXED["r_test_primary"], (R, "the frozen protocol says 79")
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])

# margin, before any prediction: the same reliability measure on both datasets
def split_half(test, t):
    w = (t >= FIXED["window_s"][0]) & (t < FIXED["window_s"][1])
    return float(noise_ceiling(test, FIXED["ceiling_splits"], FIXED["seed"], reps_scored=(test.shape[1],))["r_split_half"][:, w].mean())
rel_a = {p: split_half(bio[p][:, :R], times) for p in FIXED["included"]}
rel_e = {}
for s in FIXED["eeg2_people"]:
    e2 = te.load_part(te.subject_file(C.EEG_RAW, s, "test"))
    rel_e[s] = split_half(e2["data"], np.round(e2["times"], 3))
    del e2
DELTA = FIXED["eeg2_delta"] * float(np.median(list(rel_a.values()))) / float(np.median(list(rel_e.values())))
margin = {"alljoined_split_half": rel_a, "eeg2_split_half": rel_e, "alljoined_median": float(np.median(list(rel_a.values()))),
          "eeg2_median": float(np.median(list(rel_e.values()))), "delta": DELTA}
(C.METRICS_DIR / "09l_margin.json").write_text(json.dumps(margin, indent=1))
print("MARGIN:", {k: v for k, v in margin.items() if "median" in k or k == "delta"}, "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
ALPHAS = [10.0 ** e for e in range(-2, 9)]
n_ch, n_t = 17, FIXED["n_samples"]
win = ha.window_index(times, FIXED["map_window_ms"])
groups = window_groups(times, n_ch, C.EEG_WINDOW_MS)
query = np.concatenate([z_train, z_test])
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)

def run(people, r_test, label):
    src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
    src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
    for i, p in enumerate(people):
        pred = pl.ols_paper(z_train, y_train[p].reshape(N_TRAIN, -1), query)
        src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
    rows, failed = [], {}
    for i, p in enumerate(people):
        t = time.monotonic()
        try:
            others = [j for j in range(len(people)) if j != i]
            avg_train, avg_test = src_train[others].mean(0), src_test[others].mean(0)
            avg_t3 = t3(avg_test, N_TEST)
            preds = {"avg|all|all": avg_t3}
            for d in FIXED["draws"]:
                order = np.random.default_rng([d, p]).permutation(N_TRAIN)
                for k in FIXED["budgets"]:
                    idx = order[:k]
                    target = np.asarray(y_train[p][idx], np.float64)
                    flat = target.reshape(k, -1)
                    a_k = t3(avg_train[idx], k)
                    coefs, _ = h4b.fit_map(a_k, target, win, seed=d)
                    preds[f"h4b|{k}|{d}"] = h4b.apply_map(coefs, avg_t3, win)
                    borrowed = src_train[np.ix_(others, idx)]
                    gu_test = t3(gu_by_group(borrowed, flat, src_test[others], groups, "pooled"), N_TEST)
                    gu_oof = t3(pm.out_of_fold(lambda tr, he: gu_by_group(borrowed[:, tr], flat[tr], borrowed[:, he], groups, "pooled"),
                                               k, FIXED["oof_folds"], d), k)
                    preds[f"gu|{k}|{d}"] = gu_test
                    preds[f"gu+map|{k}|{d}"] = h4b.apply_map(h4b.fit_map(gu_oof, target, win, seed=d)[0], gu_test, win)
                    preds[f"procrustes|{k}|{d}"] = al.apply_rotation(al.fit_procrustes_map(a_k, target, win), avg_t3, win)
                    preds[f"own_ridge|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(z_train[idx], flat).predict(z_test)
            preds = {n: np.asarray(v).reshape(N_TEST, n_ch, n_t) for n, v in preds.items()}
            if not all(np.isfinite(v).all() for v in preds.values()):
                raise ValueError("nonfinite prediction")
            scored = paper_correlation(preds, bio[p][:, :r_test], n_iter=100)
            for name, curve in scored["correlation"].items():
                method, k, d = name.split("|")
                row = {"analysis": label, "person": p, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean())}
                row.update({f"r_w{w}": float(curve[:, win == w].mean()) for w in np.unique(win)})
                rows.append(row)
        except Exception as error:  # P-6: a failure is excluded and reported, never imputed
            failed[p] = f"{type(error).__name__}: {error}"
        pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"09l_scores_{label}.csv", index=False)
        print(label, f"sub-{p:02d}", f"{time.monotonic() - t:.0f} s", failed.get(p, ""), "|", f"{(time.monotonic() - T0) / 60:.1f} min")
    return pd.DataFrame(rows), failed

def decide(frame, people, failed):
    m = frame.groupby(["person", "method", "k"])["r_100_200ms"].mean()
    done = [p for p in people if p not in failed]
    def score(p, method, k):
        return m[(p, method, "all")] if method == "avg" else m[(p, method, str(k))]
    names = list(FIXED["claims"])
    d = np.array([[np.mean([score(p, a, k) - score(p, b, k) for k in FIXED["budgets"]]) for a, b in FIXED["claims"].values()] for p in done])
    out = dec.decide(d, DELTA, FIXED["alpha"], FIXED["min_people"], comparators=tuple(names))
    out["claims"] = {}
    for j, c in enumerate(names):
        pc = out["per_comparator"].get(c, {})
        verdict = ("supported" if pc.get("lower", -1) > DELTA else "rejected" if pc.get("upper", 1) < DELTA else "inconclusive") if pc else "insufficient"
        out["claims"][c] = {"compares": FIXED["claims"][c], **pc, "verdict": verdict, "sign_flip_p": dec.sign_flip_p(d[:, j])}
    out["d"] = {int(p): row.tolist() for p, row in zip(done, d)}
    out["method_means"] = {f"{meth}|{k}": float(np.mean([score(p, meth, k) for p in done])) for meth in ("avg", "h4b", "gu", "gu+map", "procrustes", "own_ridge") for k in FIXED["budgets"]}
    out["failed"] = failed
    return out

primary_frame, primary_failed = run(FIXED["included"], R, "primary")
decision = {"protocol": FIXED["protocol"], "margin": margin, "r_test": R, "primary": decide(primary_frame, FIXED["included"], primary_failed)}
(C.METRICS_DIR / "09l_decision.json").write_text(json.dumps(decision, indent=2, default=float))
for c, v in decision["primary"]["claims"].items():
    print("VERDICT", c, v["compares"], v["verdict"], "| mean", round(v.get("mean", float("nan")), 4), "lower", round(v.get("lower", float("nan")), 4),
          "| delta", round(DELTA, 4), "|", v.get("positive_targets"), "of", decision["primary"]["n_targets"])

In [ ]:
# secondary, descriptive: all 20 people, R = the smallest minimum over all 20; same delta; not part of the family
R_all = min(n_rep_min.values())
secondary_frame, secondary_failed = run(FIXED["all"], R_all, "all20")
decision["secondary_all20"] = {"r_test": R_all, **decide(secondary_frame, FIXED["all"], secondary_failed)}
(C.METRICS_DIR / "09l_decision.json").write_text(json.dumps(decision, indent=2, default=float))
print(json.dumps({c: (v["verdict"], round(v.get("mean", float("nan")), 4)) for c, v in decision["secondary_all20"]["claims"].items()}))

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-09l-outputs", title="XSUB 09l Alljoined confirmation outputs",
                  files=[C.METRICS_DIR / n for n in ("09l_decision.json", "09l_margin.json", "09l_scores_primary.csv", "09l_scores_all20.csv")],
                  include_dirs=[], message="alljoined confirmation (P-6)")